# Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model('groq:qwen/qwen3-32b')

In [5]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    name:str = Field(description="Name of the movie")
    year:str = Field(description="The year when the movie was released")
    director:str = Field(description="The name of the director")
    rating:float = Field(description="The IMDB rating of the movie")

In [6]:
model_structured = model.with_structured_output(Movie)
response = model_structured.invoke("Provide description of the movie Inception")
response

Movie(name='Inception', year='2010', director='Christopher Nolan', rating=8.8)

In [7]:
# Message output alongside parsed structure

from pydantic import BaseModel, Field

class Movie(BaseModel):
    "A description about the movie."
    name:str = Field(..., description="Name of the movie")
    year:str = Field(..., description="The year when the movie was released")
    director:str = Field(..., description="The name of the director")
    rating:float = Field(..., description="The IMDB rating of the movie")

model_structured = model.with_structured_output(Movie, include_raw=True)
response = model_structured.invoke("Provide description of the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for a description of the movie Inception. Let me check if I have the right tool for that. The available tool is the Movie function, which requires name, year, director, and rating. I need to make sure I have all that information for Inception. Let me recall: Inception was directed by Christopher Nolan, released in 2010. The IMDB rating is around 8.8. So I can use the Movie function here. I should structure the tool call with those parameters. Let me double-check the details to avoid any mistakes. Yep, that's correct. Now, format the JSON as specified.\n", 'tool_calls': [{'id': 'zrqr1qr5n', 'function': {'arguments': '{"director":"Christopher Nolan","name":"Inception","rating":8.8,"year":"2010"}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 183, 'prompt_tokens': 230, 'total_tokens': 413, 'completion_time': 0.301278712, 'completion_tokens_det

In [ ]:
# Nested

from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class Movie(BaseModel):
    name:str = Field(description="Name of the movie")
    year:str = Field(description="The year when the movie was released")
    director:str = Field(description="The name of the director")
    rating:float = Field(description="The IMDB rating of the movie")
    cast: list[Actor]
    budget: float | None = Field(None, description="The budget of the movie in USD")

model_structured = model.with_structured_output(Movie, include_raw=True)
response = model_structured.invoke("Provide description of the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for a description of the movie Inception. Let me check the tools available. There's a Movie function that requires parameters like name, year, director, rating, and cast. The user didn't specify the year or other details, but I know Inception was released in 2010, directed by Christopher Nolan. The main cast includes Leonardo DiCaprio, Joseph Gordon-Levitt, and others. The budget was around $160 million, and it has an 8.8 IMDB rating. I need to structure the function call with these details. Let me make sure all required fields are included: name, year, director, rating, and cast. The cast should be an array of objects with name and role. I'll include the main characters. Budget is optional, but I can add it for completeness. Alright, putting it all together in the tool_call format.\n", 'tool_calls': [{'id': 'qf90jec8m', 'function': {'arguments': '{"budget":160000000,"cast":[{"name":"Leonard